# Update instruction.jsonl with 3Di from S3

This notebook streams instruction JSONL parts from S3/MinIO, adds a top-level `3Di` field from amino-acid `output`, and uploads annotated parts back to a separate S3 prefix.

In [ ]:
from pathlib import Path

import torch

from libs.data.config import DataConfig
from libs.data.training.streaming import build_minio_s3_client, list_minio_text_parts
from libs.core.structure import ProstT5Structure3DiProvider, annotate_s3_instruction_jsonl_3di


In [ ]:
config = DataConfig.load()
bucket = config.minio.bucket_name

SOURCE_PREFIX_URI = f"s3://{bucket}/data/instruction/parts"
OUTPUT_PREFIX_URI = f"s3://{bucket}/data/instruction/parts_3di"

CACHE_DIR = Path("temp/3di_s3_cache")
CACHE_DB = Path("temp/3di_sequence_cache.sqlite3")

BATCH_SIZE = 2
OVERWRITE = False

SOURCE_PREFIX_URI, OUTPUT_PREFIX_URI


In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
if device.startswith("cuda"):
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

device


In [ ]:
s3_client = build_minio_s3_client(config)
parts = list_minio_text_parts(
    prefix_uri=SOURCE_PREFIX_URI,
    s3_client=s3_client,
    suffixes=(".jsonl",),
)

len(parts), [part.uri for part in parts[:5]]


In [ ]:
generation_kwargs = {
    "do_sample": False,
    "num_beams": 3,
    "repetition_penalty": 1.2,
}

provider = ProstT5Structure3DiProvider(
    device=device,
    generation_kwargs=generation_kwargs,
)


In [ ]:
summary = annotate_s3_instruction_jsonl_3di(
    provider=provider,
    prefix_uri=SOURCE_PREFIX_URI,
    output_prefix_uri=OUTPUT_PREFIX_URI,
    s3_client=s3_client,
    config=config,
    cache_dir=CACHE_DIR,
    cache_path=CACHE_DB,
    batch_size=BATCH_SIZE,
    overwrite=OVERWRITE,
    skip_existing=True,
    upload_manifest=True,
    progress_callback=print,
)

summary.to_dict()


After validating the annotated prefix, you can intentionally write back into the original prefix by setting `OUTPUT_PREFIX_URI = SOURCE_PREFIX_URI`, `OVERWRITE = True`, and passing `allow_in_place=True` to `annotate_s3_instruction_jsonl_3di`.